## DamageDiffusion - Colab Training

---

### Configuration

In [ ]:
GITHUB_REPO = "https://github.com/youlkar/damage-diffusion.git"
PROJECT_DIR = "/content/drive/MyDrive/DamageDiffusion"
NUM_EPOCHS = 100

### Mount Google Drive

In [ ]:
from google.colab import drive
import os
import time

# Mount with timeout handling
drive.mount('/content/drive')
os.makedirs(PROJECT_DIR, exist_ok=True)

print("Google Drive mounted")
print(f"Project directory: {PROJECT_DIR}")

# Verify connection with retry
max_retries = 3
for i in range(max_retries):
    try:
        # Test Drive connection
        test_files = os.listdir('/content/drive/MyDrive')
        print(f"Drive connection verified ({len(test_files)} items in MyDrive)")
        break
    except OSError as e:
        if i < max_retries - 1:
            print(f"Drive connection issue, retrying ({i+1}/{max_retries})")
            time.sleep(2)
        else:
            print("WARNING: Drive connection unstable. You may need to restart runtime.")

### Verify GPU

Auto detects GPU and configures optimal batch size based on VRAM.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print(f"GPU Detected: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    
    BATCH_SIZE = max(4, min(48, int(vram_gb * 1.5)))
    
    print(f"Auto-configured batch size: {BATCH_SIZE}")
    print(f"(Based on {vram_gb:.1f}GB VRAM)")
else:
    print("No GPU detected, using CPU batch size and settings")
    BATCH_SIZE = 2

### Clone Code from GitHub

First time: Clones repository  
Updates: Pulls latest changes

In [ ]:
import os

if os.path.exists(f"{PROJECT_DIR}/.git"):
    print("Repository already exists. Forcefully updating from GitHub (kid_testing branch)")
    %cd {PROJECT_DIR}
    
    print("\nCurrent State")
    !git log --oneline -1
    !git branch
    
    print("\nFetching Latest Changes")
    !git fetch origin
    
    print("\nForcing Update to Latest")
    !git checkout kid_testing || git checkout -b kid_testing origin/kid_testing
    
    print("\nForcing Update to Latest")
    !git reset --hard origin/kid_testing
    
    print("\nLatest Commit Pulled")
    !git log --oneline -1
    
    print("\nCode updated from GitHub from feature branch")
else:
    print("Cloning repository from GitHub (kid_testing branch)")
    !git clone -b kid_testing {GITHUB_REPO} {PROJECT_DIR}
    %cd {PROJECT_DIR}
    print("\nLatest Commit Cloned")
    !git log --oneline -1
    print("Code cloned from GitHub (kid_testing branch)")

if os.path.exists(f"{PROJECT_DIR}/backend"):
    print(f"\nBackend code found at {PROJECT_DIR}/backend")
else:
    print("\nBackend folder not found")
    print("Make sure you pushed the 'backend/' folder to GitHub")


### Download Dataset from HuggingFace

Downloads from: https://huggingface.co/datasets/varcoder/crack-segmentation-dataset
Same as original kaggle alternative (but requires authentication): https://www.kaggle.com/datasets/lakshaymiddha/crack-segmentation-dataset

In [ ]:
import os
from datasets import load_dataset

dataset_dir = f"{PROJECT_DIR}/data/crack_segmentation_dataset"

# install data
!pip install -q datasets pillow

# Check if dataset exists
if os.path.exists(dataset_dir) and os.path.exists(f"{dataset_dir}/train/images"):
    print("Dataset already exists in drive!")
    print(f"Location: {dataset_dir}")
    
    # Count files
    train_images = len(os.listdir(f"{dataset_dir}/train/images")) if os.path.exists(f"{dataset_dir}/train/images") else 0
    train_masks = len(os.listdir(f"{dataset_dir}/train/masks")) if os.path.exists(f"{dataset_dir}/train/masks") else 0
    test_images = len(os.listdir(f"{dataset_dir}/test/images")) if os.path.exists(f"{dataset_dir}/test/images") else 0
    test_masks = len(os.listdir(f"{dataset_dir}/test/masks")) if os.path.exists(f"{dataset_dir}/test/masks") else 0
    
    print(f"Train images: {train_images}, Train masks: {train_masks}")
    print(f"Test images: {test_images}, Test masks: {test_masks}")
    
    if train_images != train_masks:
        print(f"\nWARNING: Mismatch found {train_images - train_masks} images missing masks")
else:
    print("Downloading dataset from HuggingFace (2-3 min)\n")
    
    print("Step 1: Loading dataset metadata")
    dataset = load_dataset("varcoder/crack-segmentation-dataset")
    
    print(f"Step 2: Saving images to Google Drive")
    print(f"Train samples: {len(dataset['train'])}")
    print(f"Test samples: {len(dataset['test'])}")
    
    # Create directories
    os.makedirs(f"{dataset_dir}/train/images", exist_ok=True)
    os.makedirs(f"{dataset_dir}/train/masks", exist_ok=True)
    os.makedirs(f"{dataset_dir}/test/images", exist_ok=True)
    os.makedirs(f"{dataset_dir}/test/masks", exist_ok=True)
    
    # process train split
    print("\nProcessing train split")
    for i, item in enumerate(dataset['train']):
        # use pixel_values for image and label for mask
        img = item['pixel_values']
        mask = item['label']
        
        img.save(f"{dataset_dir}/train/images/{i:05d}.jpg")
        mask.save(f"{dataset_dir}/train/masks/{i:05d}.jpg")
        
        if (i+1) % 1000 == 0:
            print(f"Processed {i+1} images")
    
    # process test split
    print("\nProcessing test split")
    for i, item in enumerate(dataset['test']):
        img = item['pixel_values']
        mask = item['label']
        
        img.save(f"{dataset_dir}/test/images/{i:05d}.jpg")
        mask.save(f"{dataset_dir}/test/masks/{i:05d}.jpg")
    
    train_images = len(os.listdir(f"{dataset_dir}/train/images"))
    train_masks = len(os.listdir(f"{dataset_dir}/train/masks"))
    test_images = len(os.listdir(f"{dataset_dir}/test/images"))
    test_masks = len(os.listdir(f"{dataset_dir}/test/masks"))
    
    print(f"\nStep 3: Dataset saved to Google Drive!")
    print(f"Location: {dataset_dir}")
    print(f"Train: {train_images} images, {train_masks} masks")
    print(f"Test: {test_images} images, {test_masks} masks")
    print("Dataset will persist across all Colab sessions.")

### Install Dependencies

In [ ]:
print("Installing dependencies")

# try installing dependencies from local copy of requirements.txt
import os

try:
    !cp {PROJECT_DIR}/backend/requirements.txt /content/requirements.txt
    !pip install -q -r /content/requirements.txt
    !rm /content/requirements.txt
    print("Dependencies installed from local copy")
except Exception as e:
    print(f"Local copy failed, installing directly from GitHub")
    # fallback to manual installation
    !pip install -q torch>=2.0.0 torchvision>=0.15.0 numpy>=1.24.0 pillow>=9.5.0
    !pip install -q diffusers>=0.25.0 accelerate>=0.25.0 transformers>=4.35.0
    !pip install -q tensorboard>=2.15.0 tqdm>=4.66.0
    !pip install -q pytorch-fid>=0.3.0 scikit-learn>=1.3.0 scipy>=1.11.0
    !pip install -q pandas>=2.0.0 matplotlib>=3.7.0 seaborn>=0.12.0
    !pip install -q torch-fidelity torchmetrics
    print("Dependencies installed from direct specification")

### Start Training

**Training options available:**
- **fast**: 4-6 hours for 50 epochs
  - Medium model (~25M params), 500 timesteps, 50% data subset
  - Changes: Increased model params, timesteps and higher data subset - to learn conditioning better
  - Auto-detects GPU (T4/V100/A100) and optimizes batch size
- **medium**: 6-8 hours for 50 epochs (validation mode)
  - Full model (~45M params), 500 timesteps, 50% data subset
  - Use this to validate crack generation before full training
- **default**: Full quality, 8-16 hours for 100 epochs
  - Larger model (45M params), 1000 timesteps
  - Best quality but slower
  - Auto-detects GPU and optimizes batch size

All configs now use universal hardware auto-detection.

In [ ]:
import shutil
import os
from pathlib import Path
import time

# Source: Google Drive (slow, but persistent)
DRIVE_DATASET = f"{PROJECT_DIR}/data/crack_segmentation_dataset"

# Destination: Local SSD (fast, but ephemeral)
LOCAL_DATASET = "/content/crack_dataset"

# Check if already copied
if os.path.exists(LOCAL_DATASET):
    print("Dataset already exists on local SSD")
    print(f"Location: {LOCAL_DATASET}")
    
    # Verify integrity
    train_images = len(os.listdir(f"{LOCAL_DATASET}/train/images"))
    train_masks = len(os.listdir(f"{LOCAL_DATASET}/train/masks"))
    test_images = len(os.listdir(f"{LOCAL_DATASET}/test/images"))
    test_masks = len(os.listdir(f"{LOCAL_DATASET}/test/masks"))
    
    print(f"Train: {train_images} images, {train_masks} masks")
    print(f"Test: {test_images} images, {test_masks} masks")
    
    if train_images != train_masks:
        print(f"WARNING: Mismatch detected, re-copying...")
        shutil.rmtree(LOCAL_DATASET)
    else:
        print("Dataset ready for fast training!")
        
if not os.path.exists(LOCAL_DATASET):
    print("="*60)
    print("COPYING DATASET TO LOCAL SSD")
    print("="*60)
    print(f"Source: {DRIVE_DATASET}")
    print(f"Destination: {LOCAL_DATASET}")
    print(f"This will take ~2-3 minutes...")
    print(f"NOTE: This must be done each Colab session (SSD is ephemeral)")
    print("="*60)
    
    start_time = time.time()
    
    # Copy entire dataset tree
    shutil.copytree(DRIVE_DATASET, LOCAL_DATASET)
    
    elapsed = time.time() - start_time
    
    # Verify copy
    train_images = len(os.listdir(f"{LOCAL_DATASET}/train/images"))
    train_masks = len(os.listdir(f"{LOCAL_DATASET}/train/masks"))
    test_images = len(os.listdir(f"{LOCAL_DATASET}/test/images"))
    test_masks = len(os.listdir(f"{LOCAL_DATASET}/test/masks"))
    
    print(f"\n{'='*60}")
    print(f"DATASET COPY COMPLETE")
    print(f"{'='*60}")
    print(f"Time taken: {elapsed:.1f} seconds")
    print(f"Train: {train_images} images, {train_masks} masks")
    print(f"Test: {test_images} images, {test_masks} masks")
    print(f"\nLocal SSD dataset ready!")
    print(f"Expected speedup: 60-80x faster data loading")
    print(f"Training time: 3-5 minutes per epoch (vs 2 hours on Drive)")
    print(f"{'='*60}")

In [ ]:
%cd {PROJECT_DIR}/backend

# FULL MODEL TRAINING - Production Quality
# Expected time: 2.5-3.5 hours on A100 85GB (with optimized batch size)

!python train.py \
    --data_root /content/crack_dataset \
    --checkpoint_dir {PROJECT_DIR}/checkpoints_full \
    --log_dir {PROJECT_DIR}/logs_full \
    --num_workers 4


### Validate the training - View Samples

In [ ]:
from IPython.display import Image, display
import glob

# get latest samples
sample_pattern = f"{PROJECT_DIR}/samples/samples_epoch_*.png"
samples = sorted(glob.glob(sample_pattern))

if samples:
    latest = samples[-1]
    epoch = os.path.basename(latest).split('_')[-1].replace('.png', '')
    print(f"Latest samples (Epoch {epoch}):")
    display(Image(latest, width=800))
else:
    print("No samples generated yet.")
    print("Samples are generated every 5 epochs.")
    print("Wait for epoch 5, then re-run this cell.")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {PROJECT_DIR}/logs

### Check Training Progress

In [ ]:
import glob
import os

# check checkpoints in the full model directory
checkpoints = sorted(glob.glob(f"{PROJECT_DIR}/checkpoints_full/checkpoint_epoch_*.pt"))
print(f"-"*50 + "\nTraining Progress\n" + "-"*50)

if checkpoints:
    latest = checkpoints[-1]
    epoch = os.path.basename(latest).split('_')[-1].replace('.pt', '')
    print(f"Current epoch: {epoch}/{NUM_EPOCHS}")
    print(f"Progress: {int(epoch)/NUM_EPOCHS*100:.1f}%")
    print(f"Checkpoints saved: {len(checkpoints)}")
    print(f"\nLatest checkpoint: {os.path.basename(latest)}")
else:
    print("No checkpoints found yet.")
    print("Training may have just started.")

# check if best model exists
if os.path.exists(f"{PROJECT_DIR}/checkpoints_full/best_model.pt"):
    print("\nBest model saved")

# check samples
samples = glob.glob(f"{PROJECT_DIR}/samples/*.png")
if samples:
    print(f"\nSamples generated: {len(samples)}")

### Early Training Validation - Check if Cracks are Appearing

Run this cell during training to check if the model is learning to generate cracks. If no cracks visible by epoch 15-20, stop training and debug.

In [ ]:
%cd {PROJECT_DIR}/backend

import glob
import os
from IPython.display import Image, display
import matplotlib.pyplot as plt
from PIL import Image as PILImage

# Find the latest checkpoint from full model training
checkpoint_dir = f"{PROJECT_DIR}/checkpoints_full"
epoch_checkpoints = sorted(glob.glob(f"{checkpoint_dir}/checkpoint_epoch_*.pt"))

if not epoch_checkpoints:
    print("No checkpoints found yet. Training may have just started.")
    print("Wait a few minutes and re-run this cell.")
else:
    latest_checkpoint = epoch_checkpoints[-1]
    epoch_num = os.path.basename(latest_checkpoint).split('_')[-1].replace('.pt', '')
    
    print(f"Testing checkpoint from epoch {epoch_num}")
    print(f"Checkpoint: {os.path.basename(latest_checkpoint)}")
    print(f"-" * 60)
    
    # Generate test samples
    test_mask = f"{PROJECT_DIR}/data/crack_segmentation_dataset/test/masks/00001.jpg"
    test_output = f"{PROJECT_DIR}/validation_check"
    
    print(f"\nGenerating test samples...")
    !python inference.py \
        --checkpoint {latest_checkpoint} \
        --mask_path {test_mask} \
        --output_dir {test_output} \
        --num_samples 4 \
        --num_steps 50
    
    # Display results
    generated_files = glob.glob(f"{test_output}/*.png")
    if generated_files:
        print(f"\nValidation Results - Epoch {epoch_num}:")
        print(f"=" * 60)
        
        # Load and display mask + generated side by side
        mask_img = PILImage.open(test_mask).convert('L')
        gen_img = PILImage.open(generated_files[0])
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        axes[0].imshow(mask_img, cmap='gray')
        axes[0].set_title(f"Input Mask\n(Crack should be white)", fontsize=12)
        axes[0].axis('off')
        
        axes[1].imshow(gen_img)
        axes[1].set_title(f"Generated Samples - Epoch {epoch_num}\n(Cracks should be visible here!)", fontsize=12)
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Validation guidance
        print(f"\nValidation Checklist:")
        print(f"-" * 60)
        epoch_int = int(epoch_num)
        
        if epoch_int < 10:
            print(f"Epoch {epoch_num}: TOO EARLY - expect mostly noise/concrete")
            print(f"Action: Continue training, check again at epoch 10")
        elif epoch_int < 20:
            print(f"Epoch {epoch_num}: CRITICAL CHECKPOINT")
            print(f"Question: Do you see ANY faint crack patterns in the generated images?")
            print(f"")
            print(f"If YES (even faint): Model is learning! Continue to epoch 100")
            print(f"If NO (just concrete): Model failing. STOP and debug")
        elif epoch_int < 40:
            print(f"Epoch {epoch_num}: MID-TRAINING")
            print(f"Question: Are cracks clearly visible and matching mask shape?")
            print(f"")
            print(f"If YES: Excellent! Continue to completion")
            print(f"If NO: Model struggling. Consider stopping")
        else:
            print(f"Epoch {epoch_num}: LATE TRAINING")
            print(f"Cracks should be very clear by now.")
            print(f"If still no cracks: Stop training, investigate architecture")
        
        print(f"=" * 60)
    else:
        print("ERROR: No generated images found")

### Generate Images After Training - Inference

In [ ]:
%cd {PROJECT_DIR}/backend

import glob
import os
from IPython.display import Image, display

# Find best available checkpoint from full model training
checkpoint_dir = f"{PROJECT_DIR}/checkpoints_full"
best_checkpoint = f"{checkpoint_dir}/best_model.pt"
final_checkpoint = f"{checkpoint_dir}/final_model.pt"

# Priority: best_model.pt > final_model.pt > latest epoch checkpoint
if os.path.exists(best_checkpoint):
    CHECKPOINT = best_checkpoint
    print(f"Using best model: {CHECKPOINT}")
elif os.path.exists(final_checkpoint):
    CHECKPOINT = final_checkpoint
    print(f"Using final model: {CHECKPOINT}")
else:
    # Latest checkpoint
    epoch_checkpoints = sorted(glob.glob(f"{checkpoint_dir}/checkpoint_epoch_*.pt"))
    if epoch_checkpoints:
        CHECKPOINT = epoch_checkpoints[-1]
        epoch_num = os.path.basename(CHECKPOINT).split('_')[-1].replace('.pt', '')
        print(f"Using latest checkpoint from epoch {epoch_num}: {CHECKPOINT}")
    else:
        print("ERROR: No checkpoints found!")
        print("Please run training first")
        CHECKPOINT = None

if CHECKPOINT:
    TEST_MASK = f"{PROJECT_DIR}/data/crack_segmentation_dataset/test/masks/00001.jpg"
    OUTPUT_DIR = f"{PROJECT_DIR}/generated"

    !python inference.py --checkpoint {CHECKPOINT} --mask_path {TEST_MASK} --output_dir {OUTPUT_DIR} --num_samples 4 --num_steps 100

    generated = glob.glob(f"{OUTPUT_DIR}/*.png")
    if generated:
        print("\nGenerated images:")
        display(Image(generated[-1], width=800))

### Generate Synthetic images - dataset augmentation

Generate synthetic images from all test masks for dataset augmentation.

In [ ]:
%cd {PROJECT_DIR}/backend

import glob
import os
from IPython.display import Image, display

# Configuration for batch production
NUM_SAMPLES_PER_MASK = 8
NUM_INFERENCE_STEPS = 100

# Use final model from full training
CHECKPOINT = f"{PROJECT_DIR}/checkpoints_full/final_model.pt"

# Use test masks from crack segmentation dataset
MASK_DIR = f"{PROJECT_DIR}/data/crack_segmentation_dataset/test/masks"

# Output directory for synthetic dataset
OUTPUT_DIR = f"{PROJECT_DIR}/synthetic_dataset"

# Verify checkpoint exists
if not os.path.exists(CHECKPOINT):
    print(f"ERROR: Checkpoint not found at {CHECKPOINT}")
    print(f"Available checkpoints:")
    for ckpt in glob.glob(f"{PROJECT_DIR}/checkpoints_full/*.pt"):
        print(f"  - {ckpt}")
else:
    print(f"Using checkpoint: {CHECKPOINT}")
    print(f"Checkpoint size: {os.path.getsize(CHECKPOINT) / 1e6:.1f} MB")
    
# Verify masks exist
if not os.path.exists(MASK_DIR):
    print(f"\nERROR: Mask directory not found at {MASK_DIR}")
else:
    # Count masks
    mask_files = glob.glob(f"{MASK_DIR}/*.jpg") + glob.glob(f"{MASK_DIR}/*.png")
    total_masks = len(mask_files)
    total_images = total_masks * NUM_SAMPLES_PER_MASK
    
    print(f"\nBatch Production Configuration:")
    print(f"-" * 60)
    print(f"Checkpoint: final_model.pt")
    print(f"Input mask directory: test/masks")
    print(f"Number of test masks: {total_masks}")
    print(f"Samples per mask: {NUM_SAMPLES_PER_MASK}")
    print(f"Total synthetic images to generate: {total_images}")
    print(f"Inference steps: {NUM_INFERENCE_STEPS} (quality mode)")
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"-" * 60)
    
    estimated_time_min = total_masks * 0.5
    estimated_time_max = total_masks * 1.5
    print(f"\nEstimated time: {estimated_time_min:.0f}-{estimated_time_max:.0f} minutes")
    print("Starting batch generation...\n")
    
    # Run batch inference using test masks
    !python inference.py \
        --checkpoint {CHECKPOINT} \
        --mask_dir {MASK_DIR} \
        --output_dir {OUTPUT_DIR} \
        --num_samples {NUM_SAMPLES_PER_MASK} \
        --num_steps {NUM_INFERENCE_STEPS}
    
    # Count generated files
    generated_files = glob.glob(f"{OUTPUT_DIR}/*.png")
    print(f"\nGeneration Complete!")
    print(f"=" * 60)
    print(f"Total synthetic images created: {len(generated_files)}")
    print(f"Saved to: {OUTPUT_DIR}")
    print(f"Dataset ready for augmentation experiments!")
    print(f"=" * 60)
    
    # Display sample output
    if generated_files:
        print(f"\nSample output (first generated image):")
        display(Image(generated_files[0], width=800))

### Download Results (Optional can uncomment)

In [ ]:
# OPTIONAL TO DOWNLOAD RESULTS
# from google.colab import files
# import shutil
# import os


# print("Creating download archives\n")

# # get the best model
# if os.path.exists(f"{PROJECT_DIR}/checkpoints/best_model.pt"):
#     print("Downloading best model")
#     files.download(f"{PROJECT_DIR}/checkpoints/best_model.pt")
# else:
#     print("Best model not found")

# # zip all checkpoints
# if os.path.exists(f"{PROJECT_DIR}/checkpoints"):
#     print("Creating checkpoints archive")
#     shutil.make_archive('/content/checkpoints', 'zip', f"{PROJECT_DIR}/checkpoints")
#     files.download('/content/checkpoints.zip')

# # zip all generated samples
# if os.path.exists(f"{PROJECT_DIR}/samples"):
#     print("Creating samples archive")
#     shutil.make_archive('/content/samples', 'zip', f"{PROJECT_DIR}/samples")
#     files.download('/content/samples.zip')

# print("\nDownloads complete!")

### Clear Old Checkpoints (Save Drive Space)

In [ ]:
# Keep best_model.pt and final_model.pt, delete intermediate checkpoints
import glob
import os

intermediate = glob.glob(f"{PROJECT_DIR}/checkpoints_full/checkpoint_epoch_*.pt")
print(f"Found {len(intermediate)} intermediate checkpoints")
print("\nDelete them? (keeps best_model.pt and final_model.pt)")